# **Pathway Enrichment Analysis using g:Profiler**

This notebook produces the AT_MN family of artifacts 

# Import Libraries and Configurations

In [1]:
import json
import os
import sys
from urllib.parse import urljoin

import pandas as pd
import requests

# Add project root to Python's path
sys.path.append(os.path.abspath(os.path.join('..')))

from config import (
    NETWORK_DATA_DIRS,
    NETWORK_FILES,
)

# G:Profiler (GP) API endpoints of interest
GP_API_BASE_URL = 'https://biit.cs.ut.ee/gprofiler/api/'
GP_API_ENDPOINTS = {
    'data_versions': urljoin(GP_API_BASE_URL, 'util/data_versions'),
    'functional_profiling': urljoin(GP_API_BASE_URL, 'gost/profile'),
}

ORGANISM = 'hsapiens'

# Functions

In [2]:
def pathway_enrichment(df_genes):
    # List all genes/messenger RNAs in the network
    query = df_genes['id'].to_list()

    # Define the parameters for functional enrichment analysis
    parameters = {
        'organism': ORGANISM,
        'query': query,
        'sources': ['REAC'],
        'user_threshold': 0.1,
        'all_results': False,
        'significance_threshold_method': 'fdr',
    }

    # Request analysis for the g:Profile API
    response = requests.post(
        url=GP_API_ENDPOINTS['functional_profiling'],
        json=parameters,
    )
    response.raise_for_status()

    # Convert the response content to a JSON
    json_response = json.loads(response.content.decode('utf-8'))

    # Transform the JSON response result into a DataFrame
    df_enrichment = pd.json_normalize(json_response['result'])

    # Recover the enriched genes/messenger RNAs
    df_enrichment['genes'] = df_enrichment['intersections'] \
        .apply(
            lambda col: [
                genes for genes, sources in zip(query, col) if sources
            ]
        )

    # Remove the unnecessary columns
    df_enrichment = df_enrichment[['native','name','p_value','genes']]

    return df_enrichment

In [3]:
def create_pathway_related_network(group):
    # Define the path to the directory related to the group
    group_dir = (group.lower()).replace(' ', '-')
    dir_path = NETWORK_DATA_DIRS['processed'][group_dir]

    # Create a node DataFrame for the microRNA-mRNA interaction network
    file_name = NETWORK_FILES['interaction-nodes']
    df_interaction_nodes = pd.read_csv(os.path.join(dir_path, file_name))
    
    # Pathway enrichment analysis of genes/messenger RNAs
    df_genes = df_interaction_nodes.query('type == "Messenger RNA"')
    df_enrichment = pathway_enrichment(df_genes=df_genes)

    # Create a DataFrame of pathway nodes
    df_pathways = df_enrichment[['native', 'name']] \
        .rename(columns={'native': 'id', 'name': 'label'})
    df_pathways['type'] = 'Pathway'

    # Create a DataFrame of edges between genes and pathways
    df_membership_edges = df_enrichment \
        .explode(column='genes', ignore_index=True) \
        [['genes', 'native', 'p_value']] \
        .rename(columns={
            'genes': 'source',
            'native': 'target',
            'p_value': 'qvalue',
        })
    
    # Create a DataFrame of enriched gene nodes
    df_enriched_genes = df_membership_edges[['source']] \
        .rename(columns={'source': 'id'}) \
        .drop_duplicates(ignore_index=True)
    df_enriched_genes['label'] = df_enriched_genes['id']
    df_enriched_genes['type'] = 'Messenger RNA'

    # Concatenate enriched gene and pathway nodes in a single DataFrame
    df_membership_nodes = pd.concat(
        objs=[df_enriched_genes, df_pathways],
        ignore_index=True
    )

    # Store the DataFrame of edges
    file_name = 'membership-network-edges.csv'
    df_membership_edges.to_csv(os.path.join(dir_path, file_name), index=False)

    # Store the DataFrame of nodes
    file_name = 'membership-network-nodes.csv'
    df_membership_nodes.to_csv(os.path.join(dir_path, file_name), index=False)
    
    return df_membership_edges, df_membership_nodes

# Data Versions Endpoint

In [4]:
# Request the API version and status to the endpoint
response = requests.get(
    url=GP_API_ENDPOINTS['data_versions'],
    json={'organism': ORGANISM}
)

# Print the response content
try:
    parsed = json.loads(response.content.decode('utf-8'))
    print(json.dumps(parsed, indent=3))
except json.JSONDecodeError:
    print('Response is not valid JSON:')
    print(response.content.decode('utf-8'))

{
   "biomart": "Ensembl",
   "biomart_version": "114",
   "display_name": "Human",
   "genebuild": "GRCh38.p14",
   "gprofiler_version": "e114_eg62_p19_27110d83",
   "organism": "hsapiens",
   "sources": {
      "CORUM": {
         "name": "CORUM protein complexes",
         "version": "28.11.2022 Corum 4.1"
      },
      "GO:BP": {
         "name": "biological process",
         "version": "annotations: BioMart\nclasses: releases/2026-01-23"
      },
      "GO:CC": {
         "name": "cellular component",
         "version": "annotations: BioMart\nclasses: releases/2026-01-23"
      },
      "GO:MF": {
         "name": "molecular function",
         "version": "annotations: BioMart\nclasses: releases/2026-01-23"
      },
      "HP": {
         "name": "Human Phenotype Ontology",
         "version": "annotations: 03.2026\nclasses: None"
      },
      "HPA": {
         "name": "Human Protein Atlas",
         "version": "annotations: HPA website: 25-11-06\nclasses: script: 26-01-20"
 

# g:GOSt Endpoint

## Basal-like

In [5]:
# Create the gene-pathway membership network for the group
df_edges, df_nodes = create_pathway_related_network('Basal-like')

In [6]:
# Print the membership edges ordered by q-value
pd.merge(
    left=df_edges,
    right=df_nodes,
    how='inner',
    left_on='target',
    right_on='id'
) \
[['source', 'target', 'label', 'qvalue']] \
.sort_values(by='qvalue', ascending=True, ignore_index=True)

,source,target,label,qvalue
0,HAS2,REAC:R-HSA-5576892,Phase 0 - rapid depolarisation,0.038079
1,ZNF423,REAC:R-HSA-5576892,Phase 0 - rapid depolarisation,0.038079
2,NMT1,REAC:R-HSA-5576892,Phase 0 - rapid depolarisation,0.038079
3,TIA1,REAC:R-HSA-5576892,Phase 0 - rapid depolarisation,0.038079
4,SYT11,REAC:R-HSA-5576892,Phase 0 - rapid depolarisation,0.038079
5,RXRG,REAC:R-HSA-383280,Nuclear Receptor transcription pathway,0.064488
6,TP53INP2,REAC:R-HSA-383280,Nuclear Receptor transcription pathway,0.064488
7,ARHGEF3,REAC:R-HSA-383280,Nuclear Receptor transcription pathway,0.064488
8,SMARCD1,REAC:R-HSA-383280,Nuclear Receptor transcription pathway,0.064488
9,ELFN2,REAC:R-HSA-383280,Nuclear Receptor transcription pathway,0.064488


## Luminal A

In [7]:
# Create the gene-pathway membership network for the group
df_edges, df_nodes = create_pathway_related_network('Luminal A')

In [8]:
# Print the membership edges ordered by q-value
pd.merge(
    left=df_edges,
    right=df_nodes,
    how='inner',
    left_on='target',
    right_on='id'
) \
[['source', 'target', 'label', 'qvalue']] \
.sort_values(by='qvalue', ascending=True, ignore_index=True)

,source,target,label,qvalue
0,SERPINH1,REAC:R-HSA-1474244,Extracellular matrix organization,0.003062
1,ATOH8,REAC:R-HSA-1474244,Extracellular matrix organization,0.003062
2,NEDD4,REAC:R-HSA-1474244,Extracellular matrix organization,0.003062
3,LRMDA,REAC:R-HSA-1474244,Extracellular matrix organization,0.003062
4,PPP1R9A,REAC:R-HSA-1474244,Extracellular matrix organization,0.003062
...,...,...,...,...
235,PRKG1,REAC:R-HSA-9675108,Nervous system development,0.096172
236,COL6A3,REAC:R-HSA-9675108,Nervous system development,0.096172
237,CCNJL,REAC:R-HSA-9675108,Nervous system development,0.096172
238,ATP11C,REAC:R-HSA-9675108,Nervous system development,0.096172


## Luminal B

In [9]:
# Create the gene-pathway membership network for the group
df_edges, df_nodes = create_pathway_related_network('Luminal B')

In [10]:
# Print the membership edges ordered by q-value
pd.merge(
    left=df_edges,
    right=df_nodes,
    how='inner',
    left_on='target',
    right_on='id'
) \
[['source', 'target', 'label', 'qvalue']] \
.sort_values(by='qvalue', ascending=True, ignore_index=True)

,source,target,label,qvalue
0,FRK,REAC:R-HSA-9006934,Signaling by Receptor Tyrosine Kinases,0.000005
1,ENTPD1,REAC:R-HSA-162582,Signal Transduction,0.000005
2,RASGRF2,REAC:R-HSA-162582,Signal Transduction,0.000005
3,PPP6R3,REAC:R-HSA-162582,Signal Transduction,0.000005
4,ADAMTS2,REAC:R-HSA-162582,Signal Transduction,0.000005
...,...,...,...,...
473,DZIP1,REAC:R-HSA-5663202,Diseases of signal transduction by growth fact...,0.095842
474,ESR1,REAC:R-HSA-5663202,Diseases of signal transduction by growth fact...,0.095842
475,ZBTB4,REAC:R-HSA-5663202,Diseases of signal transduction by growth fact...,0.095842
476,RUNX1T1,REAC:R-HSA-5663202,Diseases of signal transduction by growth fact...,0.095842


## Normal

In [11]:
# Create the gene-pathway membership network for the group
df_edges, df_nodes = create_pathway_related_network('Normal')

In [12]:
# Print the membership edges ordered by q-value
pd.merge(
    left=df_edges,
    right=df_nodes,
    how='inner',
    left_on='target',
    right_on='id'
) \
[['source', 'target', 'label', 'qvalue']] \
.sort_values(by='qvalue', ascending=True, ignore_index=True)

,source,target,label,qvalue
0,LRRC1,REAC:R-HSA-162582,Signal Transduction,2.122938e-15
1,PARP6,REAC:R-HSA-162582,Signal Transduction,2.122938e-15
2,CPD,REAC:R-HSA-162582,Signal Transduction,2.122938e-15
3,ZBTB8B,REAC:R-HSA-162582,Signal Transduction,2.122938e-15
4,PRLR,REAC:R-HSA-162582,Signal Transduction,2.122938e-15
...,...,...,...,...
7052,LDLRAD3,REAC:R-HSA-450341,Activation of the AP-1 family of transcription...,9.530389e-02
7053,MAPK8,REAC:R-HSA-450341,Activation of the AP-1 family of transcription...,9.530389e-02
7054,HIF1AN,REAC:R-HSA-168799,Neurotoxicity of clostridium toxins,9.530389e-02
7055,RAB3IP,REAC:R-HSA-425381,Bicarbonate transporters,9.530389e-02
